In [2]:
# Load the libreries and the data
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import time

load_dotenv()

# Load raw data
raw_online_retail = pd.read_excel('../data/raw/Online_Retail.xlsx')

In [3]:
# Lowercase columns (same as notebook 1)
raw_online_retail.columns = raw_online_retail.columns.str.lower()

print(f"Raw data loaded: {raw_online_retail.shape[0]:,} rows")

Raw data loaded: 541,909 rows


In [4]:
# Exclude rows with zero price, no description, no customer (stock adjustments)
mask_adjustment = (
    (raw_online_retail['quantity'] < 0) &
    (raw_online_retail['unitprice'] == 0) &
    (~raw_online_retail['invoiceno'].astype(str).str.startswith('C'))
)

online_retail = raw_online_retail[~mask_adjustment].copy()
print(f"After removing stock adjustments: {online_retail.shape[0]:,} rows")
print(f"Removed: {mask_adjustment.sum():,} rows")

After removing stock adjustments: 540,573 rows
Removed: 1,336 rows


In [5]:
# Flag cancelled invoices as returns
online_retail['is_return'] = online_retail['invoiceno'].astype(str).str.startswith('C').astype(int)

print(f"Regular sales: {(online_retail['is_return'] == 0).sum():,}")
print(f"Returns: {(online_retail['is_return'] == 1).sum():,}")

Regular sales: 531,285
Returns: 9,288


In [6]:
# Cast customerID float to string and fill nulls 
online_retail['customerid'] = (online_retail['customerid'].fillna(0).astype(int).astype(str).replace('0', 'UNKNOWN'))

# Fill null descriptions
online_retail['description'] = online_retail['description'].fillna('No description')

print(f"Null CustomerID remaining: {online_retail['customerid'].isna().sum()}")
print(f"Null Description remaining: {online_retail['description'].isna().sum()}")
print(f"\nSample CustomerID values: {online_retail['customerid'].sample(3).to_list()}")

Null CustomerID remaining: 0
Null Description remaining: 0

Sample CustomerID values: ['16283', '18235', '14390']


In [7]:
# Keep zero proces only on returns, exclude on regular sales
mask_bad_price = (online_retail['unitprice'] <= 0) & (online_retail['is_return'] == 0)

print(f"Regular sales with invalid price: {mask_bad_price.sum():,}")

online_retail = online_retail[~mask_bad_price].copy()
print(f"After removing invalid prices: {online_retail.shape[0]:,} rows")

Regular sales with invalid price: 1,181
After removing invalid prices: 539,392 rows


In [8]:
online_retail['total_revenue'] = (online_retail['quantity'] * online_retail['unitprice']).round(2)

print(f"Revenue range: {online_retail['total_revenue'].min():,.2f} -> {online_retail['total_revenue'].max():,.2f}")
print(f"\nFinal clean dataset: {online_retail.shape[0]:,} rows")

Revenue range: -168,469.60 -> 168,469.60

Final clean dataset: 539,392 rows


In [9]:
# Find the symmetric transactions
top_revenue = online_retail[online_retail['total_revenue'] == 168469.60]
top_return = online_retail[online_retail['total_revenue'] == -168469.60]

print("Max revenue transaction:")
print("-" * 80)
print(top_revenue[['invoiceno', 'customerid', 'description', 'quantity', 'unitprice', 'total_revenue']])
print("\n\nMatching return:")
print("-" * 80)
print(top_return[['invoiceno', 'customerid', 'description', 'quantity', 'unitprice', 'total_revenue']])

Max revenue transaction:
--------------------------------------------------------------------------------
       invoiceno customerid                  description  quantity  unitprice  \
540421    581483      16446  PAPER CRAFT , LITTLE BIRDIE     80995       2.08   

        total_revenue  
540421       168469.6  


Matching return:
--------------------------------------------------------------------------------
       invoiceno customerid                  description  quantity  unitprice  \
540422   C581484      16446  PAPER CRAFT , LITTLE BIRDIE    -80995       2.08   

        total_revenue  
540422      -168469.6  


### Key Finding — Anomalous Transaction
Customer 16446 placed and immediately cancelled an order of 80,995 units 
of "Paper Craft, Little Birdie" worth 168,469.60 (invoices 581483 -> C581484).
This represents the largest single transaction and return in the dataset.
Flagged for exclusion from revenue KPIs as a likely data anomaly.

---

Creating the tables for MYSQL server

In [10]:
# generate one row per unique date in the dataset
dates = pd.DataFrame({'full_date': pd.date_range(
    start= online_retail['invoicedate'].min().date(),
    end= online_retail['invoicedate'].max().date(),
    freq= 'D'
)})

dates['date_key'] = dates['full_date'].dt.strftime('%Y%m%d').astype(int)
dates['year'] = dates['full_date'].dt.year
dates['quarter'] = dates['full_date'].dt.quarter
dates['month'] = dates['full_date'].dt.month
dates['month_name'] = dates['full_date'].dt.strftime('%B')
dates['week'] = dates['full_date'].dt.isocalendar().week.astype(int)
dates['day_of_month'] = dates['full_date'].dt.day
dates['day_of_week'] = dates['full_date'].dt.dayofweek + 1  # 1=Mon, 7=Sun
dates['day_name'] = dates['full_date'].dt.strftime('%A')
dates['is_weekend'] = (dates['day_of_week'] >= 6).astype(int)
dates['is_holiday'] = 0

print(f"dim_date: {len(dates):,} rows")
print(dates.head(5))

dim_date: 374 rows
   full_date  date_key  year  quarter  month month_name  week  day_of_month  \
0 2010-12-01  20101201  2010        4     12   December    48             1   
1 2010-12-02  20101202  2010        4     12   December    48             2   
2 2010-12-03  20101203  2010        4     12   December    48             3   
3 2010-12-04  20101204  2010        4     12   December    48             4   
4 2010-12-05  20101205  2010        4     12   December    48             5   

   day_of_week   day_name  is_weekend  is_holiday  
0            3  Wednesday           0           0  
1            4   Thursday           0           0  
2            5     Friday           0           0  
3            6   Saturday           1           0  
4            7     Sunday           1           0  


In [11]:
dim_geography = (
    online_retail[['country']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_geography['geography_key'] = dim_geography.index + 1
dim_geography['region'] = None
dim_geography['iso_code'] = None

print(f"dim_geography: {len(dim_geography):,} rows")
print(dim_geography.head())

dim_geography: 38 rows
          country  geography_key region iso_code
0  United Kingdom              1   None     None
1          France              2   None     None
2       Australia              3   None     None
3     Netherlands              4   None     None
4         Germany              5   None     None


In [12]:
# Obtain the region and iso_code


In [13]:
dim_product = (
    online_retail[['stockcode', 'description', 'unitprice']]
    .sort_values('unitprice', ascending=False) # A lo mejor aqui esta el problema
    .drop_duplicates(subset='stockcode', keep='first')
    .reset_index(drop=True)
)

dim_product['product_key'] = dim_product.index + 1
dim_product['category'] = None
dim_product['source'] = 'uci'
dim_product = dim_product.rename(columns={'unitprice': 'unit_price_ref'})

print(f"dim_product: {len(dim_product):,} rows")
print(dim_product.head())

dim_product: 3,938 rows
   stockcode      description  unit_price_ref  product_key category source
0          M           Manual        38970.00            1     None    uci
1  AMAZONFEE       AMAZON FEE        17836.46            2     None    uci
2          B  Adjust bad debt        11062.06            3     None    uci
3       POST          POSTAGE         8142.75            4     None    uci
4        DOT   DOTCOM POSTAGE         4505.17            5     None    uci


For project purposes, if the average order quantity per customer is more than 100 units, we consider that customer like a B2B and if the average order quantity is less than 100 is going to be a B2C.

In [14]:
# Calculate average order quantity per customer
avg_quantity = (
    online_retail[online_retail['is_return'] == 0]
    .groupby('customerid')['quantity'].mean().reset_index()
    .rename(columns= {'quantity': 'avg_quantity'})
)

dim_customer = (
    online_retail[['customerid', 'country']]
    .drop_duplicates(subset='customerid', keep='first')
    .reset_index(drop=True)
)

dim_customer['customer_key'] = dim_customer.index + 1
dim_customer['first_order_date'] = (
    online_retail.groupby('customerid')['invoicedate']
    .min()
    .dt.date
    .reset_index(drop=True)
)

# merge average quantity and assign segment
dim_customer = dim_customer.merge(avg_quantity, on= 'customerid', how= 'left')
dim_customer['segment'] = dim_customer['avg_quantity'].apply(
    lambda x: 'B2B' if x > 100 else 'B2C'
)

dim_customer = dim_customer.drop(columns= 'avg_quantity')

# Fix UNKNOWN customer row
mask_unknown = dim_customer['customerid'] == 'UNKNOWN'
dim_customer.loc[mask_unknown, 'first_order_date'] = None
dim_customer.loc[mask_unknown, 'country'] = None
dim_customer.loc[mask_unknown, 'segment'] = 'unknown'

print(f"Real customers: {(~mask_unknown).sum():,}")
print(f"UNKNOWN placeholder: {mask_unknown.sum()}")
print("\nSegment distribution:")
print(dim_customer['segment'].value_counts())
print(f"\n{dim_customer.head()}")

Real customers: 4,371
UNKNOWN placeholder: 1

Segment distribution:
segment
B2C        4274
B2B          97
unknown       1
Name: count, dtype: int64

  customerid         country  customer_key first_order_date segment
0      17850  United Kingdom             1       2011-01-18     B2C
1      13047  United Kingdom             2       2010-12-07     B2C
2      12583          France             3       2010-12-16     B2C
3      13748  United Kingdom             4       2011-11-21     B2C
4      15100  United Kingdom             5       2011-02-02     B2C


In [15]:
# Internal codes START with a letter
dim_product['is_internal'] = dim_product['stockcode'].str.match(r'^[^0-9]', na=False)

print(f"Internal codes flagged: {dim_product['is_internal'].sum()}")
print(f"Real products: {(dim_product['is_internal'] == 0).sum():,}")
print(f"\nInternal codes found:")
print(dim_product[dim_product['is_internal'] == 1][['stockcode', 'description', 'unit_price_ref']].reset_index(drop= True).sort_values(by= 'stockcode'))

Internal codes flagged: 24
Real products: 3,914

Internal codes found:
       stockcode                         description  unit_price_ref
1      AMAZONFEE                          AMAZON FEE       17836.460
2              B                     Adjust bad debt       11062.060
7   BANK CHARGES                        Bank Charges        1050.150
9             C2                            CARRIAGE         150.000
6           CRUK                     CRUK Commission        1100.440
5              D                            Discount        1867.860
22      DCGS0003                 BOXED GLASS ASHTRAY           2.510
14      DCGS0004          HAYNES CAMPER SHOULDER BAG          16.630
16      DCGS0069               OOH LA LA DOGS COLLAR          15.790
17      DCGS0070               CAMOUFLAGE DOG COLLAR          12.720
15      DCGS0076        SUNJAR LED NIGHT NIGHT LIGHT          16.130
20      DCGSSBOY                      BOYS PARTY BAG           3.290
19     DCGSSGIRL                

In [16]:
codes = online_retail[online_retail['stockcode'].str.startswith(('DCGS', 'gift'), na= False)]
print(f"'DCGS' and 'gift' transactions: {len(codes['stockcode'].unique()):,}")
print(f"Unique customers: {codes['customerid'].nunique():,}")

'DCGS' and 'gift' transactions: 12
Unique customers: 1


In [17]:
dim_product.loc[
    dim_product['stockcode'].str.startswith(('DCGS', 'gift'), na=False),
    'is_internal'
] = False

print(f"Internal codes flagged: {dim_product['is_internal'].sum()}")
print(f"Real products: {(dim_product['is_internal'] == 0).sum():,}")
print(f"\nInternal codes found:")
print(dim_product[dim_product['is_internal'] == 1][['stockcode', 'description', 'unit_price_ref']].reset_index(drop= True).sort_values(by= 'stockcode'))

Internal codes flagged: 12
Real products: 3,926

Internal codes found:
       stockcode                 description  unit_price_ref
1      AMAZONFEE                  AMAZON FEE       17836.460
2              B             Adjust bad debt       11062.060
7   BANK CHARGES                Bank Charges        1050.150
9             C2                    CARRIAGE         150.000
6           CRUK             CRUK Commission        1100.440
5              D                    Discount        1867.860
4            DOT              DOTCOM POSTAGE        4505.170
0              M                      Manual       38970.000
11          PADS  PADS TO MATCH ALL CUSHIONS           0.001
3           POST                     POSTAGE        8142.750
8              S                     SAMPLES         570.000
10             m                      Manual           2.550


In [18]:
# Build fact_sales by joining online_retail with dimension keys
fact_sales = online_retail.copy()

# Add date_key
fact_sales['date_key'] = fact_sales['invoicedate'].dt.strftime('%Y%m%d').astype(int)

# Add product_key via merge
fact_sales = fact_sales.merge(
    dim_product[['stockcode', 'product_key']],
    on='stockcode',
    how='left'
)

# Add customer_key via merge
fact_sales = fact_sales.merge(
    dim_customer[['customerid', 'customer_key']],
    on='customerid',
    how='left'
)

# Add geography_key via merge
fact_sales = fact_sales.merge(
    dim_geography[['country', 'geography_key']],
    on='country',
    how='left'
)

# Select only the columns the fact table needs
fact_sales = fact_sales[[
    'date_key', 'product_key', 'customer_key', 'geography_key',
    'invoiceno', 'quantity', 'unitprice', 'total_revenue', 'is_return'
]].copy()

fact_sales['source'] = 'uci'

print(f"fact_sales: {len(fact_sales):,} rows")
print(f"Null keys check:")
print(f"  product_key nulls:  {fact_sales['product_key'].isna().sum():,}")
print(f"  customer_key nulls: {fact_sales['customer_key'].isna().sum():,}")
print(f"  geography_key nulls:{fact_sales['geography_key'].isna().sum():,}")
print(f"  date_key nulls:     {fact_sales['date_key'].isna().sum():,}")

fact_sales: 539,392 rows
Null keys check:
  product_key nulls:  0
  customer_key nulls: 0
  geography_key nulls:0
  date_key nulls:     0


In [23]:
dim_product = dim_product.rename(columns={'stockcode': 'stock_code'})

# Rename columns to match MySQL schema exactly
dim_customer = dim_customer.rename(columns={
    'customerid': 'customer_id',
    'first_order_date': 'first_seen_date'
})

fact_sales = fact_sales.rename(columns= {
    'invoiceno': 'invoice_no',
    'unitprice': 'unit_price'
})

In [20]:
# Database connection
user     = os.getenv("MYSQL_USER")
password = os.getenv("MYSQL_PASSWORD")
host     = os.getenv("DWH_HOST")
port     = os.getenv("DWH_PORT")
db       = os.getenv("DWH_DB")

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}")

with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Connection OK' AS status, DATABASE() AS current_db, VERSION() AS mysql_version"))
    row = result.fetchone()
    print(f"Status:        {row[0]}")
    print(f"Database:      {row[1]}")
    print(f"MySQL version: {row[2]}")

Status:        Connection OK
Database:      retail_dwh
MySQL version: 8.0.46


In [ ]:
# Load dimensions first, then fact table
tables = {
    'dim_date':      dates,
    'dim_geography': dim_geography,
    'dim_product':   dim_product,
    'dim_customer':  dim_customer,
    'fact_sales':    fact_sales
}

for table_name, df_table in tables.items():
    start = time.time()
    df_table.to_sql(
        name=table_name,
        con=engine,
        if_exists='append',
        index=False
    )
    elapsed = round(time.time() - start, 2)
    print(f"{table_name}: {len(df_table):,} rows loaded in {elapsed}s")

print("\nAll tables loaded successfully!")

dim_date: 374 rows loaded in 0.03s
dim_geography: 38 rows loaded in 0.01s
dim_product: 3,938 rows loaded in 0.17s
dim_customer: 4,372 rows loaded in 0.15s
fact_sales: 539,392 rows loaded in 27.5s

All tables loaded successfully!
